In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "utils" / "paths.py").exists()
)

sys.path.insert(0, str(PROJECT_ROOT))

from utils.paths import MODEL_PATH, TEST_DATA

In [3]:
import pandas as pd
import joblib


# Load model
model = joblib.load(
    MODEL_PATH
)

# -----------------------------
# Machine Input
# -----------------------------

machine = {
    "Air_temperature_K": 300.0,
    "Process_temperature_K": 310.0,
    "Rotational_speed_rpm": 1500,
    "Torque_Nm": 40.0,
    "Tool_wear_min": 100,

    # Machine type = M
    "Type_H": 0,
    "Type_L": 0,
    "Type_M": 1
}

df = pd.DataFrame([machine])


# -----------------------------
# Feature Engineering
# -----------------------------

df["Temperature_Difference"] = (
    df["Process_temperature_K"]
    - df["Air_temperature_K"]
)

df["Power"] = (
    df["Torque_Nm"]
    * df["Rotational_speed_rpm"]
)

df["Torque_ToolWear"] = (
    df["Torque_Nm"]
    * df["Tool_wear_min"]
)


# -----------------------------
# Feature Order
# -----------------------------

feature_columns = [

    "Air_temperature_K",
    "Process_temperature_K",
    "Rotational_speed_rpm",
    "Torque_Nm",
    "Tool_wear_min",

    "Type_H",
    "Type_L",
    "Type_M",

    "Temperature_Difference",
    "Power",
    "Torque_ToolWear"
]


df = df[feature_columns]


# -----------------------------
# Prediction
# -----------------------------

probability = model.predict_proba(df)[0][1]

prediction = 1 if probability >= 0.5 else 0


# -----------------------------
# Risk Level
# -----------------------------

if probability >= 0.70:

    risk = "HIGH"

    recommendation = (
        "Maintenance required immediately."
    )

elif probability >= 0.40:

    risk = "MEDIUM"

    recommendation = (
        "Schedule maintenance soon."
    )

else:

    risk = "LOW"

    recommendation = (
        "Machine condition looks normal."
    )


# -----------------------------
# Result
# -----------------------------

print("\n================== Machine Prediction ==================")

print(f"Failure Probability: {probability * 100:.2f}%")

print(f"Prediction: {prediction}")

print(f"Risk: {risk}")

print(f"Recommendation: {recommendation}")


================== Machine Prediction ==================
Failure Probability: 0.23%
Prediction: 0
Risk: LOW
Recommendation: Machine condition looks normal.
